# Milestone 2 — Hierarchical Parsing, Standardization and Deduplication

**CSC14119 — Introduction to Data Science · Group 3**

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DuyNhat1705/Intro2DS-Group3-Source/blob/main/src/parser.ipynb)

This notebook runs the pipeline of `parsing.md` with the `src/m2` package: **gather the compilation path → preprocess (macros, conditionals) → build the hierarchy → clean and normalize math → split sentences → extract BibTeX (`.bib` and `\bibitem`) → unify and deduplicate references, rename `\cite` keys → deduplicate elements across versions → write the `elements`, `hierarchy` and `bibtex` tables and the statistics**.

The input is the Milestone 1 corpus. The notebook does not download from arXiv again, never reads `references.json`, and makes no network call while parsing. Edit the configuration cell, then choose **Run all**. Set `SAMPLE_N` to try a small sample before the full run.

In [ ]:
# CONFIGURATION — edit before Run all
from pathlib import Path

RANGES = {
    1: ("2609", 27834, 32833),
    2: ("2609", 17834, 22833),
    3: ("2608", 14007, 19006),
    4: ("2608", 4007, 9006),
    5: ("2511", 6709, 11708),
}
RUN_RANGES = "all"          # [3], [1, 2, 3] (one run, one worker "R1-2-3"), "all" or "local" (data folder)
SAMPLE_N = None           # first N papers of the selected scope; None = all
WORKER = None             # None: named after the ranges (e.g. "R4"); use different names for parallel runs

REPO_DIR = None           # local: found from the current folder; or an absolute path
REPO_URL = "https://github.com/DuyNhat1705/Intro2DS-Group3-Source.git"
BRANCH = "main"           # branch cloned on Colab (must contain src/m2)
INSTALL_DEPENDENCIES = None  # None: install on Colab, only check locally
MOUNT_DRIVE = True
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/Intro2DS-Group3"
DATA_DIR_OVERRIDE = None     # e.g. r"C:/Intro2DS-Group3-Source/data"
OUTPUT_DIR_OVERRIDE = None   # default statistics_data/parser, separate from the scraper outputs
COPY_DATA_TO_LOCAL = True    # Colab: extract one tar per selected range from Drive to /content/data
DATA_TAR_NAME = "m2_data{r}.tar"  # tar of range r in DRIVE_PROJECT_DIR; built from data/ when missing

RUN_TESTS = True             # unit tests + gold-set evaluation (about one minute)
RUN_PIPELINE = True
RESUME = True                # skip papers whose checkpoint signature is unchanged; final benchmark: False
NUM_WORKERS = "auto"         # "auto" = min(CPUs, free RAM / per-worker estimate); or an integer
TIMEOUT_SEC = 60             # per-paper deadline; on timeout the worker is killed and the paper retried once in safe mode
N_SHARDS = 8                 # Parquet shards per table after compaction
RUN_ANYSTYLE = False         # refine low-confidence bibitems with AnyStyle (needs Ruby); off by default

## 1. Running locally and on Google Colab

**Local:** use Python 3.12 and install `pip install -r src/requirements.txt` (or `conda env create -f src/intro2ds-env.yml`); open the notebook from the repository's `src/` folder or set `REPO_DIR`. The data is read from `<repo>/data` and the outputs are written to `<repo>/statistics_data/parser`. To process every paper of another folder, set `DATA_DIR_OVERRIDE` and `RUN_RANGES = "local"`.

**Colab:** choose a **CPU** runtime (the official benchmark testbed). The notebook clones the repository if needed, installs the libraries of `src/requirements.txt` and mounts Drive. By default it reads `MyDrive/Intro2DS-Group3/data` and writes `MyDrive/Intro2DS-Group3/statistics_data/parser`. Reading tens of thousands of small files directly from Drive is very slow, so the corpus is read from **one tar per range** in `MyDrive/Intro2DS-Group3`: `m2_data1.tar`, `m2_data2.tar`, ... (`DATA_TAR_NAME`). For every range of `RUN_RANGES` the notebook extracts its tar to `/content/data` (several ranges are merged there and parsed in one run); a missing tar is built once from `MyDrive/Intro2DS-Group3/data` with only the papers of that range (without `references.json`). The layout inside a tar does not matter (`data/<key>/v1/...`, `<key>/v1/...` or `R3/data_1/<key>/v1/...`): the folder that holds the paper folders is found automatically, and the Milestone 1 state of each tar is kept apart. When the tars exist, the `data/` folder on Drive is not needed. A table reports, per range, the tar, its size, the papers found and the creation/extraction times; a tar without any paper of its range stops the run. Each tar is extracted once per runtime (a replaced tar is extracted again).

**Resuming after a disconnection:** tables, checkpoints and logs are written directly to the output folder (on Drive in Colab). Every worker writes a Parquet shard every 200 papers, then records the signature of each paper in the manifest (`.state/<WORKER>/manifest`). The signature covers the relative paths, sizes and content hashes of the `.tex`/`.bib` files plus the parser version, so the Drive copy and the extracted copy share checkpoints, and a change of code or data re-parses only the affected papers. Reconnect and **Run all** to continue; at most the last unflushed batch of each worker is redone. An incomplete tar or an interrupted extraction is never reused.

**After changing the code:** restart the kernel/runtime, then Run all.

In [ ]:
import importlib.metadata
import os
import shutil
import subprocess
import sys
import time

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if REPO_DIR is not None:
    REPO_ROOT = Path(REPO_DIR).expanduser().resolve()
elif IN_COLAB:
    REPO_ROOT = Path("/content/Intro2DS-Group3-Source")
    if not REPO_ROOT.exists():
        subprocess.run(["git", "clone", "--branch", BRANCH, "--depth", "1", REPO_URL, str(REPO_ROOT)], check=True)
else:
    candidates = [Path.cwd(), *Path.cwd().parents]
    REPO_ROOT = next((p for p in candidates if (p / "src" / "m2" / "pipeline.py").is_file()), None)
    if REPO_ROOT is None:
        raise FileNotFoundError("Repository (src/m2) not found. Open the notebook inside the repository or set REPO_DIR.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

if INSTALL_DEPENDENCIES or (INSTALL_DEPENDENCIES is None and IN_COLAB):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_ROOT / "src" / "requirements.txt")],
                   check=True)

def sh(*args, check=True):
    """Run a command from the repository root with this kernel's Python and stream its output."""
    cmd = [str(a) for a in args]
    print("$", " ".join(cmd), flush=True)
    env = dict(os.environ, PYTHONIOENCODING="utf-8")
    p = subprocess.Popen(cmd, cwd=REPO_ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                         encoding="utf-8", errors="replace", env=env)
    for line in p.stdout:
        print(line, end="")
    p.wait()
    if check and p.returncode:
        raise RuntimeError(f"Command failed (exit code {p.returncode})")
    return p.returncode

import pandas as pd
from IPython.display import display
dependency_rows = []
for name in ("pyarrow", "pandas", "psutil", "tqdm", "rapidfuzz", "matplotlib", "pytest"):
    try:
        dependency_rows.append({"package": name, "version": importlib.metadata.version(name)})
    except importlib.metadata.PackageNotFoundError:
        dependency_rows.append({"package": name, "version": "MISSING"})
display(pd.DataFrame(dependency_rows))
print("Python", sys.version.split()[0], "| Colab:", IN_COLAB, "| repository:", REPO_ROOT)

In [ ]:
import re

if IN_COLAB and MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

project_root = Path(DRIVE_PROJECT_DIR) if IN_COLAB and MOUNT_DRIVE else REPO_ROOT
DATA_DIR = Path(DATA_DIR_OVERRIDE) if DATA_DIR_OVERRIDE else project_root / "data"
OUTPUT_DIR = Path(OUTPUT_DIR_OVERRIDE) if OUTPUT_DIR_OVERRIDE else project_root / "statistics_data" / "parser"

if RUN_RANGES == "local":
    selected_ranges = []
else:
    selected_ranges = sorted(RANGES) if RUN_RANGES == "all" else list(RUN_RANGES)
    if not selected_ranges or any(r not in RANGES for r in selected_ranges):
        raise ValueError("RUN_RANGES must be a list of valid ranges, 'all' or 'local'.")


def in_range(key, r):
    month, _, num = key.partition("-")
    return num.isdigit() and month == RANGES[r][0] and RANGES[r][1] <= int(num) <= RANGES[r][2]


NON_PAPER_DIRS = {"tables", "run_logs", "report_exports", "m2"}
PAPER_KEY_RE = re.compile(r"\d{4}-\d{4,5}")


def find_paper_root(top, max_depth=4):
    """Folder that directly holds the paper folders (YYMM-NNNNN) of an extracted tar, whatever
    the layout inside the tar: data/<key>/..., <key>/... or R3/data_1/<key>/..."""
    level = [top]
    for _ in range(max_depth + 1):
        nxt = []
        for d in level:
            subs = [q for q in d.iterdir() if q.is_dir()]
            if any(PAPER_KEY_RE.fullmatch(q.name) for q in subs):
                return d
            nxt += subs
        level = nxt
    return None


def build_range_tar(r, tar_path):
    """Pack the paper folders of range r (plus the M1 state) from the data folder on Drive."""
    if not DATA_DIR.is_dir():
        raise FileNotFoundError(f"{tar_path} not found, and no data folder {DATA_DIR} to build it from.")
    names = sorted(e.name for e in os.scandir(DATA_DIR) if e.is_dir() and in_range(e.name, r))
    if not names:
        raise FileNotFoundError(f"{tar_path} not found, and {DATA_DIR} has no paper of range {r} {RANGES[r]}.")
    members = [f"{DATA_DIR.name}/{n}" for n in names]
    if (DATA_DIR / ".state").is_dir():
        members.append(f"{DATA_DIR.name}/.state")
    listing = Path("/content") / f"m2_tar_R{r}.lst"
    listing.write_text("\n".join(members) + "\n", encoding="utf-8")
    part = tar_path.with_name(tar_path.name + ".part")   # renamed only when complete
    # only references.json is left out; a bare --exclude=tables would also drop v1/tables/*.tex
    subprocess.run(["tar", "-cf", str(part), "--exclude=*/references.json", "-C", str(DATA_DIR.parent),
                    "-T", str(listing)], check=True)
    os.replace(part, tar_path)


def extract_range_tar(r, tar_path, local_data):
    """Extract one tar into a staging folder, then move its paper folders into local_data."""
    stage = Path("/content/m2_stage") / f"R{r}"
    shutil.rmtree(stage, ignore_errors=True)
    stage.mkdir(parents=True)
    subprocess.run(["tar", "-xf", str(tar_path), "-C", str(stage), "--exclude=*/references.json"], check=True)
    root = find_paper_root(stage)
    if root is None:
        raise RuntimeError(f"{tar_path.name}: no paper folder (YYMM-NNNNN) inside the tar.")
    for q in root.iterdir():
        if not q.is_dir() or q.name.startswith(".") or q.name in NON_PAPER_DIRS:
            continue
        dst = local_data / q.name
        if dst.exists():
            shutil.rmtree(dst)
        os.replace(q, dst)
    if (root / ".state").is_dir():
        # M1 state of each scraper worker, prefixed with the tar name so that tars never overwrite each other
        for w in (root / ".state").iterdir():
            if w.is_dir():
                dst = local_data / ".state" / f"{tar_path.stem}__{w.name}"
                shutil.rmtree(dst, ignore_errors=True)
                dst.parent.mkdir(exist_ok=True)
                os.replace(w, dst)
    shutil.rmtree(stage, ignore_errors=True)


timing = {}
if IN_COLAB and COPY_DATA_TO_LOCAL and selected_ranges:
    local_data = Path("/content/data")
    local_data.mkdir(exist_ok=True)
    tar_rows = []
    for r in selected_ranges:
        tar_path = Path(DRIVE_PROJECT_DIR) / DATA_TAR_NAME.format(r=r)
        t0 = time.perf_counter()
        created = not tar_path.exists()
        if created:
            build_range_tar(r, tar_path)
        timing[f"tar_create_R{r}_s"] = time.perf_counter() - t0
        st = tar_path.stat()
        # one marker per exact tar (size + mtime): a replaced tar is extracted again,
        # an interrupted extraction never leaves a marker
        marker = Path("/content") / f".m2_data_R{r}_{st.st_size}_{int(st.st_mtime)}"
        t0 = time.perf_counter()
        if not marker.exists():
            for old in Path("/content").glob(f".m2_data_R{r}_*"):
                old.unlink()
            extract_range_tar(r, tar_path, local_data)
            marker.touch()
        timing[f"tar_extract_R{r}_s"] = time.perf_counter() - t0
        n = sum(1 for e in os.scandir(local_data) if e.is_dir() and in_range(e.name, r))
        tar_rows.append({"range": f"R{r}", "tar": tar_path.name, "size (GB)": round(st.st_size / 2**30, 2),
                         "built now": created, "papers of the range": n,
                         "IDs in range": RANGES[r][2] - RANGES[r][1] + 1,
                         "create (s)": round(timing[f"tar_create_R{r}_s"], 1),
                         "extract (s)": round(timing[f"tar_extract_R{r}_s"], 1)})
        if n == 0:
            raise RuntimeError(f"{tar_path.name} holds no paper of range {r} {RANGES[r]}: wrong tar for this range?")
    display(pd.DataFrame(tar_rows))
    DATA_DIR = local_data
elif not DATA_DIR.is_dir():
    raise FileNotFoundError(f"Data folder {DATA_DIR} not found. Check Drive or DATA_DIR_OVERRIDE.")
if OUTPUT_DIR.resolve() == DATA_DIR.resolve():
    raise ValueError("OUTPUT_DIR must differ from DATA_DIR to keep the parser outputs apart from the scraper sources.")

WORKER_NAME = WORKER or ("R" + "-".join(map(str, selected_ranges)) if selected_ranges else "local")
TABLES_DIR = OUTPUT_DIR / "tables" / WORKER_NAME
REPORTS_DIR = OUTPUT_DIR / "report_exports" / WORKER_NAME
LOGS_DIR = OUTPUT_DIR / "run_logs" / WORKER_NAME
STATE_DIR = OUTPUT_DIR / ".state" / WORKER_NAME
FIGURES_DIR = REPORTS_DIR / "figures"
for d in (TABLES_DIR, REPORTS_DIR, LOGS_DIR, STATE_DIR, FIGURES_DIR):
    d.mkdir(parents=True, exist_ok=True)
print("Input:", DATA_DIR)
print("Output:", OUTPUT_DIR, "| worker:", WORKER_NAME)

## 2. Scope and input check

The ranges are the ones of `scraper.ipynb`. `SAMPLE_N` keeps the first papers (by paper key) of the scope; a sample changes only the scope of the report and must not be read as the whole corpus.

The versions of each paper come from `metadata.json`, the Milestone 1 state (`data/.state/*/source_state*.json`) and the folders on disk, including the copies created by Google Drive (`v1(1)/`, `main(1).tex`). A version whose folder holds no `.tex` file is recorded as `empty`; a version listed in metadata/state without a folder is recorded as `missing`. No version is skipped.

In [ ]:
from src.m2.discover import discover, list_paper_dirs

range_specs = [f"{RANGES[r][0]}:{RANGES[r][1]}-{RANGES[r][2]}" for r in selected_ranges]
def in_ranges(key):
    month, _, num = key.partition("-")
    return not selected_ranges or (num.isdigit() and any(
        month == RANGES[r][0] and RANGES[r][1] <= int(num) <= RANGES[r][2] for r in selected_ranges))

t0 = time.perf_counter()
keys = [p.name for p in list_paper_dirs(DATA_DIR) if in_ranges(p.name)]
if SAMPLE_N is not None:
    keys = keys[:SAMPLE_N]
jobs = discover(DATA_DIR, papers=keys)
timing["input_check_s"] = time.perf_counter() - t0
n_versions = sum(len(j.versions) for j in jobs)
n_disk = sum(vi.on_disk for j in jobs for vi in j.versions)
expected = sum(RANGES[r][2] - RANGES[r][1] + 1 for r in selected_ranges)
display(pd.DataFrame([{
    "scope": ", ".join(f"R{r}" for r in selected_ranges) or "local",
    "IDs in range": expected or None, "papers with a folder": len(jobs), "versions": n_versions,
    "versions with a folder": n_disk, "versions without a folder": n_versions - n_disk,
    ".tex/.bib size (MB)": round(sum(j.tex_bytes for j in jobs) / 2**20, 1),
}]))
del jobs

## 3. Tests and gold-set evaluation

Unit tests use small inline LaTeX/BibTeX snippets in `src/m2/tests`. The gold sets are Python modules checked by hand on real papers outside the development sample: 29 versions (root, compilation path, 1,197 headings), 379 sentence boundaries, 152 `\bibitem`s and 97 reference pairs.

Targets:
- root and compilation path 100 %;
- headings ≥ 95 %;
- sentence-splitting F1 ≥ 0.97;
- bibitem: title ≥ 90 %, year ≥ 95 %;
- reference deduplication: precision ≥ 98 %, recall ≥ 95 %.

In [ ]:
if RUN_TESTS:
    sh(sys.executable, "-m", "pytest", "src/m2/tests", "-q", "-p", "no:cacheprovider")
    sh(sys.executable, "-m", "src.m2.tests.eval_gold", "--data-dir", DATA_DIR,
       "--json", REPORTS_DIR / "gold_eval.json")

## 4. Running the pipeline with checkpoints

Each worker is a separate process that handles one whole paper (all its versions) in a fixed order:

**gather → preprocess → hierarchy → cleanup/math → sentences** (per version) → **BibTeX extraction** (all versions) → **BibTeX dedup and key map** → **`\cite` renaming** → **element dedup** → **rows**.

- The main process runs a watchdog: a paper that exceeds `TIMEOUT_SEC` seconds gets its worker killed and replaced, and the paper is retried once in safe mode (macro expansion and conditional evaluation off; a success is recorded as `partial`).
- Large papers are dispatched first (LPT) to avoid a long tail.
- `NUM_WORKERS="auto"` picks the number of workers from the CPUs and the free RAM (cgroup limits included on Colab). A worker above the RSS limit flushes its work and is replaced.
- Each worker writes its own Parquet shards; only small statistics travel through the pipe. At the end the shards are merged into `N_SHARDS` deterministic files sorted by `arxiv_id`.

For the final benchmark on **Colab CPU**, set `RESUME=False` to measure one full parsing pass.

In [ ]:
if RUN_PIPELINE:
    args = [sys.executable, "-m", "src.m2.run", "--data-dir", DATA_DIR, "--out", TABLES_DIR,
            "--work-dir", STATE_DIR, "--log-dir", LOGS_DIR, "--workers", NUM_WORKERS,
            "--timeout", TIMEOUT_SEC, "--no-progress"]
    if range_specs:
        args += ["--ranges", *range_specs]
    if SAMPLE_N is not None:
        args += ["--limit", SAMPLE_N]
    if RESUME:
        args += ["--resume"]
    if RUN_ANYSTYLE:
        if IN_COLAB:
            subprocess.run("apt-get -qq install -y ruby-full > /dev/null && gem install -q anystyle-cli",
                           shell=True, check=False)
        args += ["--refine-bibitem"]
    t0 = time.perf_counter()
    sh(*args)
    timing["parse_wall_s"] = time.perf_counter() - t0
    t0 = time.perf_counter()
    sh(sys.executable, "-m", "src.m2.run", "--out", TABLES_DIR, "--work-dir", STATE_DIR,
       "--compact-only", "--n-shards", N_SHARDS)
    timing["compaction_s"] = time.perf_counter() - t0

## 5. Structural validation and the statistics required by the statement

`validate` checks the whole output:
- every `element_id`/`parent_id` of `hierarchy` exists in `elements`;
- every version is a tree rooted at `<key>#root<v>`: one parent per node, no cycle;
- no empty leaf; formulas start with `\begin{equation}` and have balanced braces; no `\(`, `\[` or `$$` remains;
- the share of sentences with residual LaTeX and the `\cite` coverage.

Corpus-wide accuracy is estimated with the paper's own Milestone 1 metadata: abstract and title fidelity (`rapidfuzz.token_set_ratio`). `stats` writes small CSV/JSON files for the report to `report_exports/<WORKER>/`.

In [ ]:
rc = sh(sys.executable, "-m", "src.m2.validate", "--out", TABLES_DIR, "--work-dir", STATE_DIR,
        "--data-dir", DATA_DIR, "--exports-dir", REPORTS_DIR, check=False)
print("validate exit code:", rc, "(0 = no structural violation)")
sh(sys.executable, "-m", "src.m2.stats", "--out", TABLES_DIR, "--work-dir", STATE_DIR,
   "--exports-dir", REPORTS_DIR, "--log-dir", LOGS_DIR)

In [ ]:
import json
import matplotlib.pyplot as plt
pd.set_option("display.max_colwidth", 120)
summary = json.load(open(REPORTS_DIR / "summary.json", encoding="utf-8"))
val = json.load(open(REPORTS_DIR / "validation.json", encoding="utf-8"))

status = pd.read_csv(REPORTS_DIR / "status_counts.csv").set_index("status")["versions"]
cov = pd.read_csv(REPORTS_DIR / "version_coverage.csv", dtype={"arxiv_id": str})
print(f"Parsing success on versions with TeX (ok/root_fallback/partial): "
      f"{summary['success_rate_non_empty']:.4%} | failed: {summary['failed_rate_non_empty']:.4%}")
if "ok_rate_non_empty" in summary:
    print(f"Strict ok rate (no sign of lost content): {summary['ok_rate_non_empty']:.4%} | "
          f"text coverage quantiles: {summary.get('coverage_quantiles')}")
display(status.rename("versions").to_frame().T)
print("Version coverage of every publication (version_coverage.csv):")
display(cov.head(15))
print(f"{len(cov):,} publications | versions per publication: {cov['n_versions'].value_counts().sort_index().to_dict()} | "
      f"with at least one parsed version: {int(((cov.ok + cov.root_fallback + cov.partial) > 0).sum()):,} | "
      f"only empty versions: {int((cov.empty == cov.n_versions).sum()):,}")
issues = pd.read_csv(REPORTS_DIR / "failure_reasons.csv")
display(issues.sort_values("versions", ascending=False).head(10))

In [ ]:
d = summary["dedup"]
dedup = pd.DataFrame({"Before": [d["nodes_before"], d["bib_cited_before_dedup"]],
                      "After": [d["elements_after"], d["bib_after"]]},
                     index=["Elements (nodes summed over versions)", "References (cited entries)"])
dedup["Reduction"] = (1 - dedup["After"] / dedup["Before"]).map("{:.1%}".format)
display(dedup)
print(f"Raw BibTeX entries in .bib/\\bibitem (before keeping cited ones): {d['bib_raw']:,} | "
      f"leaves repeating their content inside one version: {d['leaf_repeats_within_version']:,} | "
      f"identical versions reused: {d['reused_versions']:,}")

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
status.plot.bar(ax=axes[0], title="Version status"); axes[0].set_ylabel("Versions")
cov["n_versions"].value_counts().sort_index().plot.bar(ax=axes[1], title="Versions per publication")
axes[1].set_ylabel("Publications")
dedup[["Before", "After"]].set_axis(["Elements", "References"]).plot.bar(ax=axes[2], logy=True,
                                                                         title="Before and after deduplication")
fig.tight_layout(); fig.savefig(FIGURES_DIR / "coverage_dedup.png", dpi=160, bbox_inches="tight")
plt.show(); plt.close(fig)

In [ ]:
kinds = pd.read_csv(REPORTS_DIR / "kind_counts.csv").sort_values("elements", ascending=False)
depth = pd.read_csv(REPORTS_DIR / "depth_hist.csv")
files = pd.read_csv(REPORTS_DIR / "files.csv", dtype={"arxiv_id": str})
meth = pd.read_csv(REPORTS_DIR / "bib_methods.csv")
display(kinds.set_index("kind").T)
print(f"Compiled .tex files: {int(files.n_compile_files.sum()):,} | redundant files: {int(files.n_redundant_files.sum()):,}"
      f" | missing includes: {int(files.n_missing_includes.sum()):,}")
display(files["bib_source"].value_counts().rename("versions").to_frame().T)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].barh(kinds["kind"], kinds["elements"]); axes[0].set_xscale("log"); axes[0].set_title("Elements by kind")
dd = depth.groupby("depth")["count"].sum()
axes[1].bar(dd.index, dd.values); axes[1].set_yscale("log"); axes[1].set_title("Nodes by depth")
axes[2].bar(meth["parse_method"], meth["entries"]); axes[2].set_title("BibTeX entries by extraction method")
fig.tight_layout(); fig.savefig(FIGURES_DIR / "kinds_depth_bib.png", dpi=160, bbox_inches="tight")
plt.show(); plt.close(fig)

In [ ]:
checks = {
    "structural violations": val["violations_total"],
    "sentences with residual LaTeX": f"{val['residual_latex_rate']:.4%}",
    "\\cite coverage (all papers)": f"{val['cite_coverage']:.2%}",
    "\\cite coverage (papers with a bibliography source)": f"{val['cite_coverage_papers_with_bib_source']:.2%}",
    "papers with a bibliography source": val["papers_with_bib_source"],
    "abstract found": f"{val['abstract_found_share']:.2%}",
    "abstract fidelity (median / mean)": f"{val['abstract_fidelity_median']:.1f} / {val['abstract_fidelity_mean']:.1f}",
    "title fidelity (median)": val["title_fidelity_median"],
    "\\section commands / section nodes": f"{val['section_cmds']:,} / {val['section_nodes']:,}",
}
display(pd.Series(checks, name="value").to_frame())
display(pd.Series(val["violations"], name="violations").to_frame().T)
gold_path = REPORTS_DIR / "gold_eval.json"
if gold_path.exists():
    g = json.load(open(gold_path, encoding="utf-8"))
    display(pd.Series({
        "root": g["structure"]["root_acc"], "compilation path": g["structure"]["compile_acc"],
        "headings": g["structure"]["heading_acc"], "sentence F1": g["sentences"]["rules"]["f1"],
        "bibitem title": g["bibitems"]["plain"]["title_acc"], "bibitem year": g["bibitems"]["plain"]["year_acc"],
        "reference dedup precision": g["pairs"]["default"]["precision"],
        "reference dedup recall": g["pairs"]["default"]["recall"]}, name="gold-set score").to_frame())

## 6. Performance: running time and memory footprint

- **Wall time** covers the current run: checkpoint checks, parsing and shard writing. Compaction and the tar steps are timed separately.
- **Time per publication/version** is the processing time inside the worker (reading the files + parsing).
- **Maximum RAM** is the RSS of the main process plus all workers, sampled every second.
- **Output size** counts only the three dataset tables; checkpoints and logs are reported separately.

In [ ]:
runs = [json.load(open(p, encoding="utf-8")) for p in sorted(LOGS_DIR.glob("*.json"))]
run = max(runs, key=lambda r: r.get("done", 0)) if runs else {}
perf = {k: run.get(k) for k in ("papers", "workers", "done", "skipped", "failed", "timeouts", "crashes",
                                "wall_s", "paper_ms_mean", "paper_ms_median", "paper_ms_p95",
                                "version_ms_mean", "version_ms_median", "version_ms_p95", "peak_rss_mb",
                                "peak_workers_rss_mb", "paragraph_cache_hit_rate", "reused_versions")}
perf.update({k: round(v, 1) for k, v in timing.items()})
perf["tables_size_MB"] = round(sum(f.stat().st_size for t in ("elements", "hierarchy", "bibtex")
                                   for f in (TABLES_DIR / t).glob("*.parquet")) / 2**20, 2)
perf["checkpoints_and_internal_data_MB"] = round(sum(f.stat().st_size for f in STATE_DIR.rglob("*")
                                                     if f.is_file()) / 2**20, 2)
display(pd.Series(perf, name="value").to_frame())
tm = pd.read_csv(REPORTS_DIR / "timing.csv", dtype={"arxiv_id": str})
stage = (tm[["gather_ms", "preprocess_ms", "hierarchy_ms", "bib_ms", "dedup_ms"]].sum() / 1000).rename(
    {"gather_ms": "gather", "preprocess_ms": "preprocess", "hierarchy_ms": "hierarchy + cleanup + sentences",
     "bib_ms": "BibTeX", "dedup_ms": "dedup"})
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(tm["elapsed_ms"].clip(upper=tm["elapsed_ms"].quantile(0.99)), bins=50)
axes[0].set(title="Time per version (ms, clipped at p99)", xlabel="ms", ylabel="Versions")
stage.plot.bar(ax=axes[1], title="Total time per stage (s)")
fig.tight_layout(); fig.savefig(FIGURES_DIR / "performance.png", dpi=160, bbox_inches="tight")
plt.show(); plt.close(fig)
json.dump(dict(perf, colab=IN_COLAB, cpus=os.cpu_count(), worker=WORKER_NAME),
          open(REPORTS_DIR / "benchmark.json", "w", encoding="utf-8"), indent=1)

## 7. Output tables and how to read them

```text
statistics_data/parser/
├── tables/<WORKER>/
│   ├── elements/part-c*.parquet    # element_id, arxiv_id, kind, content
│   ├── hierarchy/part-c*.parquet   # arxiv_id, version, element_id, parent_id
│   └── bibtex/part-c*.parquet      # arxiv_id, citation_key, entry_type, title, authors, year, doi, journal, url
├── report_exports/<WORKER>/
│   ├── summary.json                # success rate, counts before/after deduplication
│   ├── validation.json             # structural checks, cite coverage, abstract/title fidelity
│   ├── benchmark.json              # wall time, time per publication/version, maximum RAM, output size
│   ├── gold_eval.json              # gold-set scores
│   ├── status_counts.csv, failure_reasons.csv, version_coverage.csv, files.csv, timing.csv, ...
│   └── figures/                    # figures produced by the report cells
├── run_logs/<WORKER>/<run_id>.json # one summary per run
└── .state/<WORKER>/                # checkpoints (manifest/) and internal working data:
    ├── parse_status/               # status of every version, failure reason, file counts, per-stage times
    ├── bibtex_full/                # all BibTeX fields, source, extraction method, confidence, versions, merged keys
    ├── cite_key_map/               # (version, original key) -> canonical key
    └── paper_stats/                # counts before/after per paper
```

- `kind` and `entry_type` are dictionary-encoded; `version`/`year` are int32; `authors` is a list of strings.
- `element_id = <paper_key>#<prefix><n>` with `root<v>`, `h` (heading), `a` (abstract), `l` (list), `i` (item), `b` (block), `s` (sentence), `f` (formula), `fig`, `tab`.
- Content that is identical after cleanup shares one `element_id` across versions. Each version stays a tree: the identity is kind + content + occurrence index inside the version.
- The root has a row in `elements` but no parent edge in `hierarchy`; edges are written in pre-order.

In [ ]:
import pyarrow.parquet as pq
for name in ("elements", "hierarchy", "bibtex"):
    files_ = sorted((TABLES_DIR / name).glob("*.parquet"))
    rows = sum(pq.ParquetFile(f).metadata.num_rows for f in files_)
    print(f"\n{name}: {rows:,} rows in {len(files_)} files | {TABLES_DIR / name}")
    if files_:
        with pq.ParquetFile(files_[0]) as parquet:
            print(parquet.schema_arrow)
            batch = next(parquet.iter_batches(batch_size=6), None)
            if batch is not None:
                display(batch.to_pandas())
first = pq.ParquetFile(sorted((TABLES_DIR / "hierarchy").glob("*.parquet"))[0]).read(columns=["arxiv_id"])
example = first.column("arxiv_id")[0].as_py()
print("Tree of an example publication:", example)
sh(sys.executable, "-m", "src.m2.inspect_tree", example, "--version", 1, "--depth", 2, "--width", 100,
   "--data-dir", DATA_DIR)
print("Report exports:", REPORTS_DIR)